# Line Emission Denoising — Conditional DDPM

Per the 2026-06-18 mentor pivot: full-image denoising of line-emission velocity channels (NO
patches), channel-by-channel. This notebook trains the **conditional DDPM only** — the U-Net
baseline (V12, `unet_line_emission_continuum_best.pth`) is already trained; its metrics
(PSNR 32.95dB / SSIM 0.9857 / MSE 0.000681; M0 +69.8%±15.2% / M1 +17.5%±7.8% / M2 +20.1%±14.3%)
are the comparison target below.

**Continuum-subtraction:** each channel has the static continuum removed first — the mean of
the first/last `CONTINUUM_N` line-free channels is subtracted from every channel (dataset +
evaluation), isolating the purely kinematic line emission.

**Dataset:** line-emission FITS cubes, `(201, 600, 600)`, split at the cube level (3 RunID
groups held out for inference only). Channels sampled per cube via the Gaussian sampler
(center 100, ~75% in [50,150]). Each channel downsampled to 256×256, per-channel min-max
normalised to [0,1].

The DDPM predicts the noise added to the clean channel, conditioned on `[dirty, x_t]`,
denoised via **DDIM** (25 steps).

**Kaggle setup:** GPU on, Internet on, `Add Input` → your line-emission Dataset (FITS cubes).
The bootstrap finds it under `/kaggle/input/` and points the split at it. This notebook is
**standalone** — it clones the repo itself (Section 0), so it does not depend on Kaggle's
native GitHub-linked-notebook feature and can be pasted into any blank Kaggle kernel.

**Heads-up:** DDIM sampling a full 201-channel cube is much slower than a U-Net forward pass —
Section 11's 5-cube holdout eval is the slowest cell by far.

## 0. Bootstrap (clone repo for `src/`, locate data)

In [ ]:
import os, sys, subprocess, glob

ON_KAGGLE = os.path.exists('/kaggle')
BRANCH = 'midterm-prep'
if ON_KAGGLE:
    REPO='/kaggle/working/EXXA'; PKG=os.path.join(REPO,'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch',BRANCH,'--depth','1','https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    # pytorch-msssim for the SSIM loss; bettermoments for moment-map evaluation (Sec. 11)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','pytorch-msssim','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks'));  sys.path.insert(0, PKG)
    # find the line-emission data dir under /kaggle/input (contains run_* subfolders)
    # Fail here, loudly, if the dataset is not attached. Returning None instead
    # defers the error to the first os.path.join two cells later, where it reads as
    # "TypeError: expected str ... not NoneType" and says nothing about the cause.
    hits = (glob.glob('/kaggle/input/**/*_dirty.fits', recursive=True)
            or glob.glob('/kaggle/input/**/*dirty*.fits', recursive=True))
    if not hits:
        avail = sorted(glob.glob('/kaggle/input/*'))
        raise FileNotFoundError(
            'No *_dirty.fits found under /kaggle/input.\n'
            'The line-emission Dataset is probably not attached: '
            'Add Input -> Datasets -> your line-emission cubes.\n'
            f'Currently attached: {avail if avail else "(nothing)"}')
    DATA_DIR = os.path.dirname(os.path.dirname(hits[0]))
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'): os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../data/Line Emission Data'
print('cwd:', os.getcwd(), '| DATA_DIR:', DATA_DIR)

## 0b. Pull latest code (re-run anytime — NO kernel restart needed)

After pushing new changes to the `line-emission` branch, re-run this cell to fetch them and
hot-reload the `src/` modules. Then re-run the import cell below. No "Restart & Run All"
required.

In [ ]:
# Pull latest from the line-emission branch and hot-reload src/ (no kernel restart).
# Self-contained: works even if the bootstrap cell hasn't run in this kernel.
import os, sys, subprocess

ON_KAGGLE = os.path.exists('/kaggle')
BRANCH = 'midterm-prep'
REPO = '/kaggle/working/EXXA'

if ON_KAGGLE and os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'origin/'+BRANCH], check=True)
    print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'],
                         capture_output=True, text=True).stdout.strip())
elif ON_KAGGLE:
    print('repo not cloned yet -- run the bootstrap cell (0.) first')

# drop cached project modules so the next `import` picks up the freshly pulled code
for _m in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]
print('src.* cleared from module cache -- now re-run the imports cell below.')

## 1. Imports, device, config

In [ ]:
import time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.data.cube_split import split_cubes
from src.data.fits_cube_dataset import FITSChannelDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPU  = torch.cuda.device_count()
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('device:', device, '| GPUs:', N_GPU,
      '->', [torch.cuda.get_device_name(i) for i in range(N_GPU)] if N_GPU else 'cpu')

TARGET_SIZE   = 256     # test 256 first; raise to 300 only if VRAM allows
N_SAMPLES     = 150     # channels per cube (cube has 201; DDPM is data-hungry)

## 2. Cube-level split (3 RunID groups held out for inference only)

In [ ]:
train_cubes, val_cubes, holdout_cubes = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                                     val_fraction=0.2, seed=SEED)

## 3. Datasets + DataLoaders (full-image 256x256, per-channel norm)

In [ ]:
SUBTRACT_CONTINUUM = True     # subtract line-free continuum before training/eval
CONTINUUM_N        = 5        # channels at each end averaged for the continuum estimate
train_ds = FITSChannelDataset(train_cubes, n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED,
                              subtract_continuum=SUBTRACT_CONTINUUM, continuum_n=CONTINUUM_N)
val_ds   = FITSChannelDataset(val_cubes,   n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED,
                              subtract_continuum=SUBTRACT_CONTINUUM, continuum_n=CONTINUUM_N)
print('train items:', len(train_ds), '| val items:', len(val_ds))

## 4. DDPM configuration — objective, budget, and run scope

Three training-objective options are exposed here, all measured against each other in
section 7 rather than assumed. Each is opt-in, so `prediction_type='eps'` +
`beta_schedule='linear'` + `min_snr_gamma=0` reproduces every earlier run exactly.

| Knob | Why it might help here |
|---|---|
| `cosine` schedule | linear destroys signal by the midpoint (alpha_bar 0.078 vs cosine's 0.492) — half of training sits at timesteps carrying no usable signal. A faint line in an empty field drowns sooner than a photograph would. |
| `v` prediction | eps-prediction degenerates as SNR → 0, where echoing the input is near-optimal and nothing is learned. v behaves like eps at high noise and x0 at low noise. |
| Min-SNR-γ | low-noise timesteps carry huge SNR and dominate the gradient; clipping at γ rebalances the objective across the schedule. |

**Published reference point.** The conditional DDPM for ALMA reconstruction
([Drozdova et al. 2024](https://arxiv.org/abs/2402.10204)) used **250 sampling steps**,
averaged **20 samples**, 95M parameters and 5082 training images. This notebook runs 25
steps, 4 samples, 20.7M parameters and 1050 images — roughly two orders of magnitude less
sampling compute. That gap is the most likely reason the DDPM underperforms here, and it
is a budget limit rather than a property of the method.


In [ ]:
import shutil
from src.data.stacked_pair import StackedPairDataset
from src.models.diffusion_unet import default_diffusion_config
from src.training.diffusion import DenoisingDiffusion

# ---- budget -----------------------------------------------------------------
EPOCHS_DDPM      = 60     # full training for the chosen objective
LR_DDPM          = 2e-4
BATCH_SIZE_DDPM  = 8      # 256px DDPM is heavy; the probe below shrinks on OOM
SAMPLING_STEPS   = 25     # DDIM steps at evaluation
K_AVG            = 4      # posterior-mean: average K reverse draws
RESCALE_TO_DIRTY = False  # match each denoised channel's mean/std to the dirty channel's
                          # before inverting the normalisation -- see section 13b
DENOISE_BS_DDPM  = 8      # channels per sampling batch in the holdout evaluation

# ---- objective sweep (section 7) --------------------------------------------
RUN_SWEEP        = False  # ANSWERED TWICE: v11 and v13 both rank C_v_cosine first with a
                          # ~19 dB gap to eps+linear, and v13 persisted all four arms, so
                          # the models exist. Set True only to add a NEW objective.
SWEEP_EPOCHS     = 12     # short runs -- ranking the objective, not training it out
RUN_BASELINE_EVAL = True  # score the restored checkpoint BEFORE any training
SWEEP_EVAL_BATCH = 8      # val batches scored per sweep run (8 x batch = ~64 channels)

# ---- training view: full images vs patches ----------------------------------
# Only 6 RunIDs are trained on. Patches do not create new disks, but they turn one
# whole-image sample into many distinct local structures, which is what a denoiser has to
# model. Section 7 sweeps this as an arm rather than assuming it helps.
PATCH_SIZE       = 64
N_PATCHES        = 8      # per channel -> 1050 images become ~8400 patch samples
PATCH_SIGNAL_BIAS = 0.5   # half placed on the source; uniform wastes patches on empty sky

# ---- inference view: 256 round-trip vs native 600 ---------------------------
# The holdout currently downsamples 600 -> 256, denoises, upsamples back. TILED_NATIVE
# instead runs the model at its own tile size across the full field, no resampling.
TILED_NATIVE     = False  # section 14 compares both on the same checkpoint
TILE_OVERLAP     = 64     # Hann-blended; a hard seam would land mid-field

# ---- seeds (section 8) ------------------------------------------------------
SEEDS_DDPM       = [42]       # one seed fits an overnight session; add 43 on a second
                              # night -- section 8 resumes and appends without retraining

# Every stage below appends its rows as it finishes and skips what is already on
# record, so this notebook can be run across several sessions without losing work.
OUT_DIR = '../results'
os.makedirs(OUT_DIR, exist_ok=True)
CKPT_DIR = '../results/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)


def persist_ckpt(path, note=''):
    """Copy a finished checkpoint to /kaggle/working IMMEDIATELY, not at the end.

    CKPT_DIR lives inside the git clone, which the bootstrap wipes with `git reset --hard`
    and which is NOT part of the notebook Output. Only /kaggle/working survives. v11
    trained four sweep arms plus a full model over ~6 h and persisted them in a final cell
    that a later crash never reached, so all six were destroyed with the container -- the
    numbers survived in the log, the models did not, and the whole run had to be repeated.

    Called the moment each model finishes training, so a crash at hour 5 costs the hours
    after it and nothing before.
    """
    if not (ON_KAGGLE and path and os.path.exists(path)):
        return None
    base = os.path.basename(path)
    dst = os.path.join('/kaggle/working', base[:-4] if base.endswith('.pth.tar') else base)
    shutil.copy2(path, dst)
    print('    [persisted] {} ({:.0f} MB){}'.format(
        os.path.basename(dst), os.path.getsize(dst) / 1e6, (' -- ' + note) if note else ''),
        flush=True)
    return dst

print('DDPM budget: {} epochs | lr {} | batch {} | {} DDIM steps | K_avg {}'.format(
    EPOCHS_DDPM, LR_DDPM, BATCH_SIZE_DDPM, SAMPLING_STEPS, K_AVG))
print('sweep: {} | {} epochs/run | seeds {}'.format(RUN_SWEEP, SWEEP_EPOCHS, SEEDS_DDPM))
print()
print('Holdout cost is exactly linear in all three of:')
print('  201 channels / DENOISE_BS_DDPM  x  SAMPLING_STEPS  x  K_AVG  model calls per cube')
print('The smoke test in 10c measures one batch and projects the total before you commit.')


## 5. Datasets — stack the continuum-subtracted pairs for the DDPM

`train_ds`/`val_ds` (Section 3 above) already yield continuum-subtracted, shared-dirty-scale
`(dirty, clean)` pairs. `StackedPairDataset` just stacks each pair into the `(2,H,W)` tensor the
DDPM trainer expects — no FITS re-read, no re-normalization.

In [ ]:
from src.data.patches import PatchPairDataset, tiled_denoise, tile_grid
ddpm_train_ds = StackedPairDataset(train_ds)
ddpm_val_ds   = StackedPairDataset(val_ds)
print('DDPM train items:', len(ddpm_train_ds), '| val items:', len(ddpm_val_ds))

def make_ddpm_loaders(bs):
    nw = 4 if ON_KAGGLE else 0
    return (DataLoader(ddpm_train_ds, batch_size=bs, shuffle=True,  num_workers=nw, pin_memory=True),
            DataLoader(ddpm_val_ds,   batch_size=bs, shuffle=False, num_workers=nw, pin_memory=True))
ddpm_train_loader, ddpm_val_loader = make_ddpm_loaders(BATCH_SIZE_DDPM)


def make_patch_loaders(bs):
    """Loaders over the patch view. Batch is halved: each item already carries N_PATCHES
    patches, which _flatten_patches folds into the batch axis, so the effective batch is
    bs * N_PATCHES."""
    nw = 4 if ON_KAGGLE else 0
    b = max(1, bs // 2)
    return (DataLoader(ddpm_train_patch, batch_size=b, shuffle=True,  num_workers=nw, pin_memory=True),
            DataLoader(ddpm_val_patch,   batch_size=b, shuffle=False, num_workers=nw, pin_memory=True))
# Patch view of the same data, for the 'patch' sweep arm in section 7. Same cubes, same
# split, same continuum subtraction -- only how each channel is cut into training samples
# differs, so the arm isolates that one choice.
ddpm_train_patch = PatchPairDataset(train_ds, patch_size=PATCH_SIZE, n_patches=N_PATCHES,
                                    seed=SEED, signal_bias=PATCH_SIGNAL_BIAS)
ddpm_val_patch = PatchPairDataset(val_ds, patch_size=PATCH_SIZE, n_patches=N_PATCHES,
                                  seed=SEED, signal_bias=PATCH_SIGNAL_BIAS)
print('patch view: {} items x {} patches of {}px = {:,} training samples'.format(
    len(ddpm_train_patch), N_PATCHES, PATCH_SIZE, len(ddpm_train_patch) * N_PATCHES))
print('  (full-image view: {:,} samples of {}px)'.format(len(ddpm_train_ds), TARGET_SIZE))
_ng = len(tile_grid(600, 600, TARGET_SIZE, TILE_OVERLAP))
print('native-600 tiled inference would need {} tiles/channel at {}px overlap {}'.format(
    _ng, TARGET_SIZE, TILE_OVERLAP))


## 6. Model — conditional DDPM U-Net at 256px

`make_cfg` builds a config from an objective triple so section 7 can sweep them on equal
terms. `prediction_type` is written into the checkpoint and restored on load: a v-trained
model decoded as eps produces plausible-looking noise rather than an error, so the
objective has to travel with the weights.


In [ ]:
def make_cfg(prediction_type='eps', beta_schedule='linear', min_snr_gamma=0.0):
    """Config for one objective. Defaults reproduce every pre-existing run exactly."""
    c = default_diffusion_config(image_size=TARGET_SIZE)
    c.model.ch_mult = [1, 2, 2, 2, 4]   # 5 levels -> bottleneck at 16x16 (attn@16 fires)
    c.model.ema_rate = 0.99             # short runs: 0.999 lags too far behind
    c.diffusion.prediction_type = prediction_type
    c.diffusion.beta_schedule = beta_schedule
    c.diffusion.min_snr_gamma = float(min_snr_gamma)
    return c


# The objective actually used for the full run. Section 7 overwrites these from its
# winner; edit here directly to skip the sweep and go straight to a known-good setting.
#
# Winner of the 41df52d sweep, by a margin nothing else here comes close to:
#   C_v_cosine          v   cosine  0.0  -> PSNR 37.822   (full 60-epoch run: 38.363)
#   D_v_cosine_minsnr5  v   cosine  5.0  -> PSNR 36.326
#   patch_view          v   cosine  0.0  -> PSNR 35.966
#   A_eps_linear      eps   linear  0.0  -> PSNR 17.907   <- what every run before this used
# 19.9 dB of that spread is the objective alone, on identical data. This ALSO selects which
# checkpoint cell 6b restores, since losses are only comparable within one objective.
BEST_OBJ = dict(prediction_type='v', beta_schedule='cosine', min_snr_gamma=0.0)

cfg = make_cfg(**BEST_OBJ)
print('config:', 'ch', cfg.model.ch, '| ch_mult', cfg.model.ch_mult,
      '| attn', cfg.model.attn_resolutions, '| T', cfg.diffusion.num_diffusion_timesteps)
print('objective:', BEST_OBJ)

CKPT_DDPM = os.path.join(CKPT_DIR, 'ddpm_line_emission_continuum_best.pth.tar')

# param count + forward-shape sanity (conditional input = 2 channels [cond, x_t])
_probe = DenoisingDiffusion(config=cfg, device=str(device), lr=LR_DDPM,
                            checkpoint_path=CKPT_DDPM, data_parallel=False)
print('DDPM params:', f'{sum(p.numel() for p in _probe._core.parameters()):,}',
      '  (published ALMA reference: ~95M)')
with torch.no_grad():
    _x = torch.randn(2, 2, TARGET_SIZE, TARGET_SIZE, device=device)
    _t = torch.randint(0, cfg.diffusion.num_diffusion_timesteps, (2,), device=device).float()
    _o = _probe._core(_x, _t)
print('forward (2,2,%d,%d) -> %s' % (TARGET_SIZE, TARGET_SIZE, tuple(_o.shape)))
del _probe
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 6b. Restore the trained checkpoint from a previous session

Must run BEFORE section 7. Section 7 decides whether to train by checking whether
`CKPT_DDPM` exists on disk; when this restore sat at 10b instead, a top-to-bottom run
reached section 7 with nothing restored yet, found no checkpoint, and trained 60 epochs
from scratch (~2 h) before 10b later announced the checkpoint it had just made redundant.

Training is ~2 h; the holdout evaluation is a separate ~15-45 min and is what usually runs out
of session. Re-running the whole notebook to redo the evaluation would retrain for two hours
first, which is pure waste.

The checkpoint lives inside `/kaggle/working/EXXA`, wiped by the bootstrap's `git reset --hard`
and not carried between sessions. Section 12 copies it to the top level of `/kaggle/working`,
where it becomes the notebook's **Output**.

**To evaluate without retraining:** `Add Input` -> `Notebook Output` -> the previous version of
this notebook, then run sections 0-6, this cell, and 11. Sections 7-10 (training and its plots)
can be skipped entirely.

In [ ]:
import glob, shutil

# pull the checkpoint (and any completed per-cube results) back from a previous session
restored = 0
if ON_KAGGLE:
    # Checkpoints are chosen by LOWEST best_val_loss, not by mtime. Separate sessions
    # produce separate lineages, and the newest is not the best: one restore brought back
    # epoch 54 / best_val 19.6441 while a previous session had reached epoch 44 / 17.7247.
    # Section 11 would then have been evaluated with the worse weights, with nothing in the
    # output to show it -- "best_val went UP across epochs" is the only symptom, and it is
    # easy to read as noise.
    # Match by CONTENT, not by filename. Globbing one exact name meant a checkpoint
    # attached under any other name -- Kaggle renaming it, a nested folder, .pth instead
    # of .pth.tar -- returned zero hits and printed "no checkpoint found", which reads as
    # "nothing was attached" rather than "the name did not match".
    # '.ckpt' is here for a Kaggle-specific reason: a torch checkpoint IS a zip archive
    # internally, and Kaggle unpacks archive-like files when a DATASET is uploaded -- so a
    # '.pth' uploaded that way arrives as a DIRECTORY of data.pkl/byteorder/version and
    # torch.load fails with "Is a directory". Notebook OUTPUTS are left alone, which is why
    # the same file survives there. Uploading under a neutral extension is the workaround.
    ck_hits = sorted(set(glob.glob("/kaggle/input/**/*.pth.tar", recursive=True)) |
                     set(glob.glob("/kaggle/input/**/*.pth", recursive=True)) |
                     set(glob.glob("/kaggle/input/**/*.ckpt", recursive=True)))
    ck_hits = [h for h in ck_hits if os.path.isfile(h)]   # skip unpacked-to-directory ones
    if not ck_hits:
        print('no .pth/.pth.tar anywhere under /kaggle/input. Attached inputs:')
        for d in sorted(glob.glob('/kaggle/input/*/*/*'))[:40]:
            print('   ', d)
        if not glob.glob('/kaggle/input/*'):
            print('    (nothing attached at all -- add the dataset as an Input)')
    if ck_hits and not os.path.exists(CKPT_DDPM):
        scored = []
        for h in ck_hits:
            try:
                _c = torch.load(h, map_location='cpu', weights_only=False)
                # A U-Net checkpoint from notebook 05 is also a .pth under /kaggle/input.
                # Loading one here would restore the wrong architecture, so require the
                # DDPM's markers. NOT 'prediction_type': that key was added in 750cf38,
                # AFTER Kaggle v8 (aa2a41d) wrote the epoch-99 checkpoint, so requiring it
                # would reject exactly the file this is meant to find. 'ema_helper' plus
                # 'state_dict' is present in both eras; the U-Net writes 'model_state_dict'
                # and no 'ema_helper', so the two never collide.
                if not (isinstance(_c, dict) and 'ema_helper' in _c and 'state_dict' in _c):
                    print(f'  skipped (not a DDPM checkpoint): {h}')
                    del _c
                    continue
                # Tag with the objective it was TRAINED under. Checkpoints written before
                # 750cf38 have no such key and were all eps -- that is the correct default,
                # not a guess.
                scored.append((float(_c.get('best_val_loss', float('inf'))),
                               int(_c.get('epoch', 0)), h,
                               str(_c.get('prediction_type', 'eps'))))
                del _c
            except Exception as e:
                print('  unreadable checkpoint skipped:', h, '--', e)
        for bv, ep, h, pt in sorted(scored):
            print(f'  candidate: epoch {ep:>3} | best_val {bv:.4f} | pred {pt:<3} | {h}')

        # Compare losses ONLY within one objective. eps-loss and v-loss are on different
        # scales, so a straight minimum across both picks by scale rather than by quality:
        # the eps epoch-99 checkpoint reads best_val 13.47 against the v run's 67.18, and
        # would win outright despite scoring 17.9 dB where the v model scores 38.4. It also
        # predates 'prediction_type', so nothing would correct the config afterwards and a
        # v-configured trainer would decode eps weights as v.
        _want = BEST_OBJ['prediction_type']
        # Drop the sweep's own checkpoints. They are 12-epoch ranking runs, and the patch
        # arm's loss is summed over 64px tiles rather than 256px images, so it reads far
        # lower than a fully-trained full-image model without being better -- epoch 9 at
        # 21.97 would beat epoch 41 at 67.18. Comparable losses need the same objective AND
        # the same training view.
        _match = [t for t in scored
                  if t[3] == _want and 'sweep' not in os.path.basename(t[2])]
        if not _match:
            print(f'  NO checkpoint trained with prediction_type={_want!r} -- not restoring.')
            print(f'  (found: {sorted({t[3] for t in scored})}. Training starts from scratch;')
            print('   attach an output from a run using this objective to resume instead.)')
        if _match:
            bv, ep, h, pt = sorted(_match)[0]
            os.makedirs(os.path.dirname(CKPT_DDPM), exist_ok=True)
            shutil.copy2(h, CKPT_DDPM); restored += 1
            print(f'restored BEST {pt}-prediction checkpoint '
                  f'(epoch {ep}, best_val {bv:.4f}) from {h}')

    # The sweep CSV, so section 7 can print its table (and skip re-running arms) without
    # the 1.5 h. Restored read-only: RUN_SWEEP=False means nothing appends to it.
    _sw_dst = os.path.join(OUT_DIR, 'ddpm_objective_sweep.csv')
    _sw_hits = sorted(glob.glob('/kaggle/input/**/ddpm_objective_sweep.csv', recursive=True),
                      key=os.path.getmtime)
    if _sw_hits and not os.path.exists(_sw_dst):
        os.makedirs(os.path.dirname(_sw_dst), exist_ok=True)
        shutil.copy2(_sw_hits[-1], _sw_dst); restored += 1
        print('restored ddpm_objective_sweep.csv from', _sw_hits[-1])

    # Seed-named checkpoints, under the name section 8 rebinds CKPT_DDPM to. Section 15
    # persists them as 'ddpm_seed42.pth' while section 8 looks for 'ddpm_seed42.pth.tar',
    # so without this a resumed session reads the CSV, correctly skips training, and then
    # dies on FileNotFoundError loading weights it was told already existed.
    # '.ckpt' as well as '.pth': if the notebook Output cannot be attached -- Kaggle's Add
    # Input pins to a notebook's LATEST version, so a later failed run hides an earlier
    # good one -- the checkpoint has to come in as a Dataset instead, and a '.pth' uploaded
    # that way is unpacked into a directory torch.load rejects (RULES.md #3). Uploading it
    # renamed to '.ckpt' survives, so accept that too.
    _seed_hits = (glob.glob('/kaggle/input/**/ddpm_seed*.pth*', recursive=True) +
                  glob.glob('/kaggle/input/**/ddpm_seed*.ckpt', recursive=True))
    for _sh in sorted(set(_seed_hits)):
        if not os.path.isfile(_sh):
            continue
        _stem = os.path.basename(_sh).split('.pth')[0].split('.ckpt')[0]   # ddpm_seed42
        _sdst = os.path.join(CKPT_DIR, _stem + '.pth.tar')
        if not os.path.exists(_sdst):
            os.makedirs(CKPT_DIR, exist_ok=True)
            shutil.copy2(_sh, _sdst); restored += 1
            print(f'restored {_stem} -> {os.path.basename(_sdst)}')

    # ddpm_seed_repeats.csv too: section 8 checks it to decide whether to train. Without
    # it, a resumed session loads the epoch-58 checkpoint and trains the remaining 2 epochs
    # -- harmless but it changes the weights, so a diagnostics re-score would no longer be
    # measuring the checkpoint it is reporting on.
    _sr_dst = os.path.join(OUT_DIR, 'ddpm_seed_repeats.csv')
    _sr_hits = sorted(glob.glob('/kaggle/input/**/ddpm_seed_repeats.csv', recursive=True),
                      key=os.path.getmtime)
    if _sr_hits and not os.path.exists(_sr_dst):
        os.makedirs(os.path.dirname(_sr_dst), exist_ok=True)
        shutil.copy2(_sr_hits[-1], _sr_dst); restored += 1
        print('restored ddpm_seed_repeats.csv from', _sr_hits[-1])

    _csv_dst = '../results/moment_map_holdout_summary_ddpm.csv'
    _csv_hits = sorted(glob.glob("/kaggle/input/**/moment_map_holdout_summary_ddpm.csv",
                                 recursive=True), key=os.path.getmtime)
    if _csv_hits and not os.path.exists(_csv_dst):
        os.makedirs(os.path.dirname(_csv_dst), exist_ok=True)
        shutil.copy2(_csv_hits[-1], _csv_dst); restored += 1
        print('restored', os.path.basename(_csv_dst), 'from', _csv_hits[-1])

if os.path.exists(CKPT_DDPM):
    _ck = torch.load(CKPT_DDPM, map_location='cpu', weights_only=False)
    print(f"\ncheckpoint present: epoch {_ck.get('epoch')} | "
          f"best_val {float(_ck.get('best_val_loss', float('nan'))):.4f}")
    print('-> sections 7-10 (training) can be SKIPPED; run 10c (smoke test) then section 11')
    del _ck
else:
    print('no checkpoint found -- section 7 must run (about 2 h)')

## 6c. OOM-safe batch probe

Finds the largest batch that fits at 256px by halving on OOM. Resolution is never reduced —
the moment maps are scored at native 600px, so shrinking the training field would change
what the model learns rather than just how fast it learns it.


In [ ]:
# OOM-safe batch probe (256px fixed; shrink batch only)
def probe_ddpm_batch(bs0):
    bs = bs0
    while bs >= 1:
        try:
            dd = DenoisingDiffusion(config=cfg, device=str(device), lr=LR_DDPM, checkpoint_path=CKPT_DDPM)
            tl, _ = make_ddpm_loaders(bs)
            x, _ = next(iter(tl)); x = x.to(device)
            from src.training.diffusion import data_transform, noise_estimation_loss
            x = data_transform(x)
            e = torch.randn_like(x[:, 1:, :, :])
            t = torch.randint(0, dd.num_timesteps, (x.size(0),), device=device)
            loss = noise_estimation_loss(dd.model, x, t, e, dd.betas)
            loss.backward()
            del dd, loss, x, e; torch.cuda.empty_cache()
            return bs
        except RuntimeError as ex:
            if 'out of memory' not in str(ex).lower(): raise
            torch.cuda.empty_cache(); bs //= 2
            print(f'[OOM] reducing batch -> {bs} (image stays {TARGET_SIZE})')
    raise RuntimeError('does not fit even at batch 1')

BATCH_DDPM_USED = probe_ddpm_batch(BATCH_SIZE_DDPM)
gpu_note = f'DataParallel x{N_GPU} (~{max(1,BATCH_DDPM_USED // max(N_GPU,1))}/GPU)' if N_GPU > 1 else 'single GPU'
print(f'BATCH SIZE USED: {BATCH_DDPM_USED} at {TARGET_SIZE}x{TARGET_SIZE}  [{gpu_note}]')
ddpm_train_loader, ddpm_val_loader = make_ddpm_loaders(BATCH_DDPM_USED)

## 6c-bis. Holdout evaluation machinery

Defined once and called twice — on the restored checkpoint in 6d and on the newly
trained model in section 14. Sharing the code path is what makes those two tables
comparable: the only thing that differs between them is the weights.


In [ ]:
import csv
import gc
import matplotlib.ticker as mticker
import bettermoments as bm
from astropy.io import fits
from src.evaluation.moment_maps import (generate_moment_maps,
                                        moment_improvement)
from src.data.fits_cube_dataset import continuum_of

# OUT_DIR / DENOISE_BS_DDPM / SAMPLING_STEPS / K_AVG all come from section 4

# The denoised cubes are 289 MB each and nothing downstream reads them back -- the moment
# maps are now built from the in-memory array instead of a write-then-reread round trip.
# Turn on only if the cubes themselves are wanted as an artifact.
SAVE_DENOISED_FITS = False


def _rss_gb():
    """Resident set size in GiB, read straight from /proc (no psutil dependency)."""
    try:
        for line in open('/proc/self/status'):
            if line.startswith('VmRSS:'):
                return int(line.split()[1]) / 2**20
    except Exception:
        return float('nan')
    return float('nan')


def _mem(tag):
    # Two sessions were killed by the host OOM killer with no traceback, at the same point
    # each time. Printing RSS around every large allocation turns the next failure into a
    # measurement instead of another inference.
    print('      [mem] {:<26} RSS {:.2f} GiB'.format(tag, _rss_gb()), flush=True)


def mdiff(a, b):
    mask = np.isfinite(a) & np.isfinite(b)
    return float(np.nanmean(np.abs(a[mask] - b[mask])))

def denoise_cube_ddpm(ho_entry, velax):
    """Denoise one cube, and collapse the DIRTY moments while only that cube is resident.

    Ordering is the whole point. Each cube is 201x600x600 float32 = 289 MB and
    generate_moment_maps allocates one more internally for the M2 noise clip. Collapsing
    dirty here, before the denoised cube is allocated, means dirty_csub can be freed first
    and the two never coexist -- the previous version held dirty_csub, the denoised cube
    and a third full copy of dirty simultaneously, and the kernel was killed by the host
    OOM killer at exactly that point on two runs.
    """
    with fits.open(ho_entry['dirty'], memmap=False) as hdul:
        dirty_raw = np.ascontiguousarray(hdul[0].data).astype(np.float32)
        hdr = hdul[0].header.copy()
    C, H, W = dirty_raw.shape
    dirty_csub = dirty_raw - continuum_of(dirty_raw, CONTINUUM_N)[None, :, :]
    del dirty_raw
    gc.collect()
    _mem('dirty loaded')

    dmaps = generate_moment_maps(None, data_velax=(dirty_csub, velax))
    _mem('dirty moments done')

    los  = dirty_csub.reshape(C, -1).min(axis=1)
    his  = dirty_csub.reshape(C, -1).max(axis=1)
    rngs = his - los
    norm = np.zeros_like(dirty_csub)
    nz = rngs > 0
    norm[nz] = (dirty_csub[nz] - los[nz, None, None]) / rngs[nz, None, None]
    del dirty_csub          # freed BEFORE the denoised cube is allocated
    gc.collect()

    denoised_csub = np.empty((C, H, W), dtype=np.float32)
    _mem('norm + denoised alloc')
    # Progress matters here: one cube is 201 channels x 25 DDIM steps x K_AVG draws,
    # so the cell can sit silent for many minutes and look hung.
    _nb = -(-C // DENOISE_BS_DDPM)
    print('    {} channels in {} batches of {} ({} model calls per batch)'.format(
        C, _nb, DENOISE_BS_DDPM, SAMPLING_STEPS * K_AVG), flush=True)
    _t0 = time.time()

    def _tile_fn(batch):
        """Denoise one batch of native-resolution tiles. Used only when TILED_NATIVE."""
        return eval_diff.sample(batch.to(device), sampling_timesteps=SAMPLING_STEPS,
                                use_ema=True, n_avg=K_AVG).cpu()

    if TILED_NATIVE:
        # Run the model at its own tile size across the full 600px field -- no 600->256->600
        # round trip, which otherwise caps every number this notebook reports. Tiles are
        # Hann-blended, so no seam lands mid-field where the moments are computed.
        for ch in range(C):
            den = tiled_denoise(norm[ch], _tile_fn, tile=TARGET_SIZE,
                                overlap=TILE_OVERLAP, batch=2)
            denoised_csub[ch] = (den * rngs[ch] + los[ch]) if rngs[ch] > 0 else \
                                np.full((H, W), los[ch], np.float32)
            if (ch + 1) % 10 == 0 or ch + 1 == C:
                _el = time.time() - _t0
                print('    {:>3}/{} channels (tiled native) | {:.1f} min elapsed, '
                      '~{:.1f} min left'.format(ch + 1, C, _el / 60,
                                                _el / (ch + 1) * (C - ch - 1) / 60), flush=True)
        del norm
        gc.collect()
        _mem('sampling done (tiled native)')
        return denoised_csub, dmaps

    for s in range(0, C, DENOISE_BS_DDPM):
        t   = torch.from_numpy(norm[s:s+DENOISE_BS_DDPM])[:, None].float()          # (b,1,600,600) [0,1]
        t256 = F.interpolate(t, (TARGET_SIZE, TARGET_SIZE), mode='bilinear', align_corners=False)
        out  = eval_diff.sample(t256, sampling_timesteps=SAMPLING_STEPS, use_ema=True, n_avg=K_AVG)  # (b,1,256,256) [0,1]
        out600 = F.interpolate(out.cpu(), (H, W), mode='bilinear', align_corners=False)[:, 0].numpy()
        for k in range(out600.shape[0]):
            ch = s + k
            o = out600[k]
            if RESCALE_TO_DIRTY:
                # v13's failure: the sampler emits a narrow band around 0.5 (its own smoke
                # test recorded range [0.348, 0.701]) instead of spanning [0,1], so
                # inverting the dirty-scale normalisation lands the whole field at
                # mid-scale -- a constant pedestal over empty sky. M0 went to -310% on one
                # cube with the disk itself still recovered.
                #
                # Match the denoised channel's first two moments to the DIRTY channel's
                # before inverting. Denoising should not change a channel's mean or spread
                # much; it should remove noise from their spatial arrangement.
                sd_o = float(o.std())
                if sd_o > 0:
                    ref = norm[ch]
                    o = (o - o.mean()) / sd_o * float(ref.std()) + float(ref.mean())
            denoised_csub[ch] = (o * rngs[ch] + los[ch]) if rngs[ch] > 0 else \
                                np.full((H, W), los[ch], np.float32)
        _done = min(s + DENOISE_BS_DDPM, C)
        _el = time.time() - _t0
        _eta = _el / max(_done, 1) * (C - _done)
        # One line per batch, NOT a '\r'-rewritten status line: a carriage return is
        # rewritten in a live browser but dropped from Kaggle's saved/committed log, so
        # the only progress indicator vanished in exactly the runs that needed it -- a
        # slow-but-working run looked byte-identical to a hung one.
        print('    {:>3}/{} channels | {:.1f} min elapsed, ~{:.1f} min left'.format(
            _done, C, _el / 60, _eta / 60), flush=True)
    del norm
    gc.collect()
    _mem('sampling done, norm freed')
    if SAVE_DENOISED_FITS:
        out_path = os.path.join(OUT_DIR, 'denoised_ddpm_' + ho_entry['folder'] + '.fits')
        fits.writeto(out_path, denoised_csub, header=hdr, overwrite=True)
        print('    wrote', os.path.basename(out_path), flush=True)
    return denoised_csub, dmaps



# One field list for both writers. They were duplicated, and when moment_improvement
# started returning the unmasked M*_all values alongside the masked ones, `row` grew three
# keys that neither list knew about -- csv.DictWriter raises on extra keys, so the improved
# holdout died AFTER sampling every cube, at the write.
MOMENT_FIELDS = ['cube', 'dirty_M0', 'imp_M0', 'imp_M0_all',
                 'dirty_M1', 'imp_M1', 'imp_M1_all',
                 'dirty_M2', 'imp_M2', 'imp_M2_all']


def run_holdout(ckpt_path, obj, tag, csv_path, make_visual_maps=True):
    """Score one checkpoint on all five held-out cubes. Returns (rows, maps).

    Called twice: once in 6d on the restored checkpoint, so an unattended session yields
    moment maps by hour two, and once in section 14 on whatever section 8 trained. Both
    passes therefore share identical evaluation code -- the only difference is the weights,
    which is the whole point of running the first one.

    `obj` must be the objective the checkpoint was trained with; a v-trained model decoded
    as eps produces plausible-looking noise rather than an error. `load_checkpoint`
    overrides it from the file when the file records one.
    """
    global eval_diff, ddpm_maps

    for _stale in ('diffusion', '_probe'):
        if _stale in globals() and globals()[_stale] is not None:
            del globals()[_stale]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # data_parallel=False is load-bearing: with 2 GPUs the default re-replicates the model
    # on every forward call, and this makes SAMPLING_STEPS * K_AVG of them per batch. Two
    # sessions deadlocked there at exactly 88/201 channels before this was pinned.
    eval_diff = DenoisingDiffusion(config=make_cfg(**obj), device=str(device), lr=LR_DDPM,
                                   checkpoint_path=ckpt_path, data_parallel=False)
    eval_diff.load_checkpoint(ckpt_path)
    print('[{}] checkpoint epoch {} | best_val {:.4f} | prediction_type {}'.format(
        tag, eval_diff.start_epoch, float(eval_diff.best_val_loss),
        eval_diff.prediction_type), flush=True)
    print('[{}] evaluating {} held-out cubes (DDIM sampling -- slow)...'.format(
        tag, len(holdout_cubes)), flush=True)

    
    def _completed_cubes(path):
        """Cube rows already scored, so a resumed session skips them.
    
        Each cube is 15-45 min of DDIM sampling and the whole cell used to be
        all-or-nothing: a session that died on cube 4 threw away cubes 1-3. Rows are
        appended as each cube finishes, and MEAN/STD summary rows are ignored here."""
        out = {}
        if not os.path.exists(path):
            return out
        with open(path, newline='') as f:
            for r in csv.DictReader(f):
                cube = (r.get('cube') or '').strip()
                if cube in ('', 'MEAN', 'STD') or not r.get('imp_M0'):
                    continue
                try:
                    out[cube] = {'cube': cube,
                                 **{k: float(r[k]) for k in r if k != 'cube' and r[k] not in ('', None)}}
                except ValueError:
                    continue
        return out
    
    done_cubes = _completed_cubes(csv_path)
    if done_cubes:
        print(f'[resume] {len(done_cubes)} cube(s) already scored, will be skipped: '
              f'{sorted(done_cubes)}')
    
    ddpm_maps = None
    rows_data = []
    col_w = 30
    hdr_line = '{:<{w}} {:>10} {:>10} {:>10} {:>10} {:>10} {:>10}'.format(
        'cube', 'dirty_M0', 'imp_M0%', 'dirty_M1', 'imp_M1%', 'dirty_M2', 'imp_M2%', w=col_w)
    print('\n' + hdr_line); print('-' * len(hdr_line))
    
    for ho in holdout_cubes:
        if ho['folder'] in done_cubes:
            rows_data.append(done_cubes[ho['folder']])
            r = done_cubes[ho['folder']]
            print('  {:<24} SKIPPED (already scored)  M0 {:>7.1f}%  M1 {:>7.1f}%  M2 {:>7.1f}%'.format(
                ho['folder'], r['imp_M0'], r['imp_M1'], r['imp_M2']))
            continue
        print('  denoising', ho['folder'], '...', flush=True)
        _mem('cube start')
    
        # THE LEAK: this was `_, velax = bm.load_cube(...)`. bettermoments returns the whole
        # cube as float64 -- 201x600x600x8 = 579 MB -- and binding it to `_` kept it alive for
        # the rest of the iteration, through all three moment collapses, purely to obtain a
        # 201-element velocity axis. Only velax is wanted; drop the payload immediately.
        _cube, velax = bm.load_cube(ho['dirty'])
        del _cube
        gc.collect()
        _mem('velax read, cube freed')
    
        denoised_cube, (d0, d1, d2) = denoise_cube_ddpm(ho, velax)
    
        n0, n1, n2 = generate_moment_maps(None, data_velax=(denoised_cube, velax))
        del denoised_cube
        gc.collect()
        _mem('denoised moments done')
    
        with fits.open(ho['clean'], memmap=False) as h:
            clean_csub = np.ascontiguousarray(h[0].data).astype(np.float32)
        clean_csub -= continuum_of(clean_csub, CONTINUUM_N)[None]     # in place: no second copy
        c0, c1, c2 = generate_moment_maps(None, data_velax=(clean_csub, velax))
        del clean_csub
        gc.collect()
        _mem('clean moments done')
        row = {'cube': ho['folder']}
        # Signal-masked scoring, identical to every other notebook -- see
        # src/evaluation/moment_maps.moment_improvement. Averaging over empty sky let pixels
        # with no line dominate, which hit M2 hardest -- its denominator vanishes exactly
        # there. The mask comes from the CLEAN M0 alone, so it is identical for every method.
        _imp = moment_improvement((c0, c1, c2), (d0, d1, d2), (n0, n1, n2))
        for nm, cl, di in [('M0', c0, d0), ('M1', c1, d1), ('M2', c2, d2)]:
            row['dirty_' + nm] = round(mdiff(cl, di), 6)
            row['imp_' + nm] = round(_imp[nm], 2)
            row['imp_' + nm + '_all'] = round(_imp[nm + '_all'], 2)
        # keep this cube's maps for the visual in section 14 (and persist them, so the
        # figure survives a session that only resumes the later cells)
        if ho['folder'] == holdout_cubes[0]['folder']:
            ddpm_maps = {'clean': [c0, c1, c2], 'dirty': [d0, d1, d2], 'ddpm': [n0, n1, n2]}
            np.savez_compressed(os.path.join(OUT_DIR, 'ddpm_moment_maps.npz'),
                                c0=c0, c1=c1, c2=c2, d0=d0, d1=d1, d2=d2,
                                n0=n0, n1=n1, n2=n2)
        rows_data.append(row)
        # append as each cube finishes, so a timeout costs one cube instead of all five
        _new = not os.path.exists(csv_path)
        with open(csv_path, 'a', newline='') as _cf:
            _w = csv.DictWriter(_cf, fieldnames=MOMENT_FIELDS)
            if _new:
                _w.writeheader()
            _w.writerow(row)
        print('{:<{w}} {:>10.4g} {:>9.1f}% {:>10.4g} {:>9.1f}% {:>10.4g} {:>9.1f}%'.format(
            ho['folder'], row['dirty_M0'], row['imp_M0'], row['dirty_M1'], row['imp_M1'],
            row['dirty_M2'], row['imp_M2'], w=col_w))
    
    moments = ['M0', 'M1', 'M2']
    imps  = {m: [r['imp_'+m] for r in rows_data if r['imp_'+m] == r['imp_'+m]] for m in moments}
    means = {m: float(np.mean(imps[m])) for m in moments}
    stds  = {m: float(np.std(imps[m], ddof=1)) if len(imps[m]) > 1 else 0.0 for m in moments}
    print('\n' + '=' * len(hdr_line))
    print('DDPM SUMMARY (n=' + str(len(rows_data)) + ' cubes):')
    for m in moments:
        print('  ' + m + ': mean {:+.1f}%  std {:.1f}%  (n={})'.format(means[m], stds[m], len(imps[m])))
    
    fieldnames = MOMENT_FIELDS
    # rewritten in full here (mode 'w'), collapsing the incrementally appended rows and
    # adding MEAN/STD -- so re-running never leaves duplicates behind
    with open(csv_path, 'w', newline='') as cf:
        w = csv.DictWriter(cf, fieldnames=fieldnames); w.writeheader()
        for r in rows_data: w.writerow(r)
        w.writerow({'cube': 'MEAN', **{'imp_'+m: round(means[m],2) for m in moments},
                    **{'dirty_'+m: '' for m in moments}})
        w.writerow({'cube': 'STD',  **{'imp_'+m: round(stds[m],2) for m in moments},
                    **{'dirty_'+m: '' for m in moments}})
    print('\nCSV saved ->', csv_path)
    
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(moments))
    bar_means = [means[m] for m in moments]; bar_stds = [stds[m] for m in moments]
    colors = ['#4E91C7' if v >= 0 else '#E8715A' for v in bar_means]
    bars = ax.bar(x, bar_means, yerr=bar_stds, capsize=6, color=colors, alpha=0.85,
                  error_kw=dict(elinewidth=1.5, ecolor='#333333'))
    ax.axhline(0, color='#333333', lw=0.8, ls='--')
    ax.set_xticks(x); ax.set_xticklabels(['Moment 0\n(intensity)','Moment 1\n(velocity)','Moment 2\n(dispersion)'])
    ax.set_ylabel('Improvement over dirty (%)')
    ax.set_title('DDPM: moment-map improvement across 5 held-out cubes\n(positive = denoised closer to clean than dirty)')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%+.0f%%'))
    for bar, mv, sv in zip(bars, bar_means, bar_stds):
        ht = bar.get_height(); va = 'bottom' if ht >= 0 else 'top'
        ax.text(bar.get_x()+bar.get_width()/2, ht + (sv+2)*(1 if ht>=0 else -1),
                '{:+.1f}% +/-{:.1f}%'.format(mv, sv), ha='center', va=va, fontsize=9)
    for i, m in enumerate(moments):
        ax.scatter([i]*len(imps[m]), imps[m], color='k', s=22, zorder=5, alpha=0.7)
    ax.grid(axis='y', alpha=0.3); plt.tight_layout()
    chart_path = os.path.join(OUT_DIR, 'moment_map_holdout_summary_ddpm.png')
    plt.savefig(chart_path, dpi=140); plt.show()
    print('Chart saved ->', chart_path)
    return rows_data, ddpm_maps


## 6d. Baseline holdout — score the restored checkpoint before training anything

**This exists so an unattended run always produces something.** The sweep and the full
training below take hours; if the session dies before the holdout evaluation at the end,
the night yields nothing. This scores the checkpoint restored in 6b first, so the moment
maps exist by hour two regardless of what happens afterwards.

It is also the cleanest available comparison. The last completed run scored −4190% M0, but
that run had *two* independent faults: its checkpoint restore failed so it trained from
scratch to `best_val 16.79` (against the 13.47 already on disk), and it scored with the old
whole-map metric. This isolates both — same evaluation code as section 14, on the good
checkpoint, with the signal-masked metric.


In [ ]:
BASELINE_CSV = os.path.join(OUT_DIR, 'moment_map_holdout_baseline_ddpm.csv')

if RUN_BASELINE_EVAL and os.path.exists(CKPT_DDPM):
    _bck = torch.load(CKPT_DDPM, map_location='cpu', weights_only=False)
    _bobj = dict(BEST_OBJ)
    _bobj['prediction_type'] = _bck.get('prediction_type', 'eps')
    print('baseline checkpoint: epoch {} | best_val {:.4f} | objective {}'.format(
        _bck.get('epoch'), float(_bck.get('best_val_loss', float('nan'))),
        _bobj['prediction_type']))
    del _bck
    print('\nScoring it on the 5 held-out cubes BEFORE any training, so this session')
    print('produces moment maps even if it is killed later. ~2 h.\n', flush=True)
    baseline_rows, baseline_maps = run_holdout(CKPT_DDPM, _bobj, 'baseline', BASELINE_CSV)
else:
    baseline_rows, baseline_maps = None, None
    print('no restored checkpoint (or RUN_BASELINE_EVAL=False) -- skipping the baseline pass')


## 7. Objective sweep — which training objective actually helps?

Five objectives on short runs, ranked by **validation PSNR**, not by validation loss.
Loss is not comparable across objectives: eps and v regress different targets, and Min-SNR
rescales the loss outright, so the lowest number would simply be whichever objective has
the smallest target — a meaningless comparison. PSNR is computed the same way for all five.

Each run trains `SWEEP_EPOCHS` epochs from scratch at a fixed seed and is scored on a
capped subset of validation batches. Short runs rank objectives; they do not train any of
them out. Rows append as each finishes, so a timeout costs one run.


In [ ]:
import csv

SWEEP_CSV = os.path.join(OUT_DIR, 'ddpm_objective_sweep.csv')
SWEEP_FIELDS = ['name', 'prediction_type', 'beta_schedule', 'min_snr_gamma',
                'epochs', 'best_val_loss', 'psnr', 'ssim', 'mse', 'wall_time_s']

# Three objectives, not five: an overnight session has to reach the holdout evaluation,
# and each extra objective costs ~25 min that the moment maps need more. These three
# isolate the two changes most likely to matter -- the schedule and the prediction target
# -- against the exact configuration every previous run used.
OBJECTIVES = {
    'A_eps_linear':        dict(prediction_type='eps', beta_schedule='linear', min_snr_gamma=0.0),
    'C_v_cosine':          dict(prediction_type='v',   beta_schedule='cosine', min_snr_gamma=0.0),
    'D_v_cosine_minsnr5':  dict(prediction_type='v',   beta_schedule='cosine', min_snr_gamma=5.0),
}

# The patch arm reuses the best objective but changes the TRAINING VIEW, so it isolates
# "more, smaller samples" from "different objective". It is listed separately because it
# needs different loaders, not just a different config.
SWEEP_PATCH_ARM = True


def _sweep_done(path):
    out = {}
    if os.path.exists(path):
        with open(path, newline='') as f:
            for r in csv.DictReader(f):
                if r.get('name') and r.get('psnr') not in ('', None):
                    out[r['name']] = r
    return out


sweep_rows = _sweep_done(SWEEP_CSV)
if sweep_rows:
    print(f'[resume] {len(sweep_rows)} objective(s) already scored: {sorted(sweep_rows)}')

if RUN_SWEEP:
    for name, obj in OBJECTIVES.items():
        if name in sweep_rows:
            r = sweep_rows[name]
            print(f'--- {name}: SKIPPED, already scored (PSNR {float(r["psnr"]):.3f}) ---')
            continue
        print(f'\n{"="*70}\n=== {name}: {obj}\n{"="*70}', flush=True)
        torch.manual_seed(SEED); np.random.seed(SEED)
        c = make_cfg(**obj)
        d = DenoisingDiffusion(config=c, device=str(device), lr=LR_DDPM,
                               checkpoint_path=os.path.join(CKPT_DIR, f'ddpm_sweep_{name}.pth.tar'),
                               data_parallel=False)
        t0 = time.time()
        d.train(ddpm_train_loader, ddpm_val_loader, n_epochs=SWEEP_EPOCHS, log_every=4)
        # PSNR, not loss -- see the note above.
        m = d.evaluate(ddpm_val_loader, sampling_timesteps=SAMPLING_STEPS,
                       max_batches=SWEEP_EVAL_BATCH, use_ema=True, n_avg=K_AVG)
        row = {'name': name, **obj, 'epochs': SWEEP_EPOCHS,
               'best_val_loss': round(float(d.best_val_loss), 5),
               'psnr': round(float(m['psnr']), 4), 'ssim': round(float(m['ssim']), 5),
               'mse': round(float(m['mse']), 8), 'wall_time_s': round(time.time() - t0, 1)}
        sweep_rows[name] = row
        persist_ckpt(SWEEP_CSV)
        new = not os.path.exists(SWEEP_CSV)
        with open(SWEEP_CSV, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=SWEEP_FIELDS)
            if new:
                w.writeheader()
            w.writerow({k: row.get(k, '') for k in SWEEP_FIELDS})
        print('  -> PSNR {:.3f} | SSIM {:.4f} | best_val {:.4f} | {:.0f}s'.format(
            row['psnr'], row['ssim'], row['best_val_loss'], row['wall_time_s']))
        persist_ckpt(os.path.join(CKPT_DIR, f'ddpm_sweep_{name}.pth.tar'), name)
        del d
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    # ---- patch arm: same objective as the best so far, different training view ----
    if SWEEP_PATCH_ARM and 'patch_view' not in sweep_rows:
        _base = (max(sweep_rows.values(), key=lambda r: float(r['psnr']))
                 if sweep_rows else {'prediction_type': 'eps', 'beta_schedule': 'linear',
                                     'min_snr_gamma': 0.0})
        _pobj = dict(prediction_type=_base['prediction_type'],
                     beta_schedule=_base['beta_schedule'],
                     min_snr_gamma=float(_base['min_snr_gamma']))
        print(f'\n{"="*70}\n=== patch_view: {_pobj} on {PATCH_SIZE}px patches\n{"="*70}',
              flush=True)
        torch.manual_seed(SEED); np.random.seed(SEED)
        _pl, _pv = make_patch_loaders(BATCH_DDPM_USED)
        d = DenoisingDiffusion(config=make_cfg(**_pobj), device=str(device), lr=LR_DDPM,
                               checkpoint_path=os.path.join(CKPT_DIR, 'ddpm_sweep_patch.pth.tar'),
                               data_parallel=False)
        t0 = time.time()
        d.train(_pl, _pv, n_epochs=SWEEP_EPOCHS, log_every=4)
        # Scored on the FULL-IMAGE validation loader, not the patch one: the deliverable is
        # a whole denoised channel, and a patch-trained model that only looked good on
        # patches would be measuring its own training view.
        m = d.evaluate(ddpm_val_loader, sampling_timesteps=SAMPLING_STEPS,
                       max_batches=SWEEP_EVAL_BATCH, use_ema=True, n_avg=K_AVG)
        row = {'name': 'patch_view', **_pobj, 'epochs': SWEEP_EPOCHS,
               'best_val_loss': round(float(d.best_val_loss), 5),
               'psnr': round(float(m['psnr']), 4), 'ssim': round(float(m['ssim']), 5),
               'mse': round(float(m['mse']), 8), 'wall_time_s': round(time.time() - t0, 1)}
        sweep_rows['patch_view'] = row
        with open(SWEEP_CSV, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=SWEEP_FIELDS).writerow(
                {k: row.get(k, '') for k in SWEEP_FIELDS})
        print('  -> PSNR {:.3f} | SSIM {:.4f} | {:.0f}s'.format(
            row['psnr'], row['ssim'], row['wall_time_s']))
        persist_ckpt(os.path.join(CKPT_DIR, 'ddpm_sweep_patch.pth.tar'), 'patch_view')
        del d, _pl, _pv
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    print('RUN_SWEEP is False -- using BEST_OBJ from section 6 as-is.')

if sweep_rows:
    print('\n' + '=' * 86)
    print('{:<24}{:>8}{:>10}{:>10}{:>12}{:>12}'.format(
        'objective', 'pred', 'schedule', 'minSNR', 'PSNR', 'SSIM'))
    print('-' * 86)
    for name, r in sorted(sweep_rows.items(), key=lambda item: -float(item[1]['psnr'])):
        print('{:<24}{:>8}{:>10}{:>10}{:>12.3f}{:>12.4f}'.format(
            name, r['prediction_type'], r['beta_schedule'], r['min_snr_gamma'],
            float(r['psnr']), float(r['ssim'])))
    print('=' * 86)
    _best = max(sweep_rows.values(), key=lambda r: float(r['psnr']))
    BEST_OBJ = dict(prediction_type=_best['prediction_type'],
                    beta_schedule=_best['beta_schedule'],
                    min_snr_gamma=float(_best['min_snr_gamma']))
    cfg = make_cfg(**BEST_OBJ)
    print('\nwinner ->', _best['name'], BEST_OBJ)
    _spread = max(float(r['psnr']) for r in sweep_rows.values()) - \
              min(float(r['psnr']) for r in sweep_rows.values())
    print('spread across objectives: {:.2f} dB'.format(_spread))
    print('These are {}-epoch runs at ONE seed. Notebook 08 measured ~1.0 dB of pure seed'
          .format(SWEEP_EPOCHS))
    print('spread on the U-Net, so treat a gap below ~1 dB here as unresolved, not as a win.')


## 8. Train the winning objective — full budget, multiple seeds

The sweep ranks objectives on short runs; this trains the winner properly. Two seeds give
the headline number an error bar, which the whole project has learned to insist on: the
U-Net sweep's apparent "+4.16 dB winner" turned out to sit inside a ±1.0 dB seed band, and
a single DDPM number would be exactly as unfalsifiable.

Each seed checkpoints separately and is skipped if already on record.


In [ ]:
SEED_CSV = os.path.join(OUT_DIR, 'ddpm_seed_repeats.csv')
SEED_FIELDS = ['seed', 'prediction_type', 'beta_schedule', 'min_snr_gamma',
               'epochs_run', 'best_val_loss', 'psnr', 'ssim', 'mse', 'wall_time_s']


def _seed_done(path):
    out = {}
    if os.path.exists(path):
        with open(path, newline='') as f:
            for r in csv.DictReader(f):
                try:
                    out[int(r['seed'])] = r
                except (KeyError, ValueError):
                    continue
    return out


seed_rows = _seed_done(SEED_CSV)
# A recorded row is only a skip if its weights are actually present. The CSV and the
# checkpoints are restored separately, so they can disagree -- and a row without weights
# makes this section skip training and then fail loading what it just claimed to have.
_orphaned = [sd for sd in seed_rows
             if not os.path.exists(os.path.join(CKPT_DIR, f'ddpm_seed{sd}.pth.tar'))]
for sd in _orphaned:
    print(f'[resume] seed {sd} is in the CSV but its checkpoint is missing -- will retrain')
    del seed_rows[sd]
if seed_rows:
    print(f'[resume] seeds already trained: {sorted(seed_rows)}')

for sd in SEEDS_DDPM:
    ck = os.path.join(CKPT_DIR, f'ddpm_seed{sd}.pth.tar')
    if sd in seed_rows:
        r = seed_rows[sd]
        print(f'--- seed {sd}: SKIPPED, already done (PSNR {float(r["psnr"]):.3f}) ---')
        continue
    print(f'\n{"="*70}\n=== seed {sd} | {BEST_OBJ}\n{"="*70}', flush=True)
    torch.manual_seed(sd); np.random.seed(sd)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(sd)
    d = DenoisingDiffusion(config=make_cfg(**BEST_OBJ), device=str(device), lr=LR_DDPM,
                           checkpoint_path=ck, data_parallel=False)
    if os.path.exists(ck):
        d.load_checkpoint(ck)      # resume a seed interrupted mid-training
    _todo = max(0, EPOCHS_DDPM - d.start_epoch)
    t0 = time.time()
    if _todo:
        d.train(ddpm_train_loader, ddpm_val_loader, n_epochs=_todo, log_every=1)
    m = d.evaluate(ddpm_val_loader, sampling_timesteps=SAMPLING_STEPS,
                   use_ema=True, n_avg=K_AVG)
    row = {'seed': sd, **BEST_OBJ, 'epochs_run': d.start_epoch,
           'best_val_loss': round(float(d.best_val_loss), 5),
           'psnr': round(float(m['psnr']), 4), 'ssim': round(float(m['ssim']), 5),
           'mse': round(float(m['mse']), 8), 'wall_time_s': round(time.time() - t0, 1)}
    seed_rows[sd] = row
    persist_ckpt(ck, f'seed {sd}')
    new = not os.path.exists(SEED_CSV)
    with open(SEED_CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=SEED_FIELDS)
        if new:
            w.writeheader()
        w.writerow({k: row.get(k, '') for k in SEED_FIELDS})
    print('  seed {}: PSNR {:.3f} | SSIM {:.4f}'.format(sd, row['psnr'], row['ssim']))
    del d
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if seed_rows:
    _ps = [float(r['psnr']) for r in seed_rows.values()]
    print('\n=== DDPM over {} seed(s) ==='.format(len(_ps)))
    print('  PSNR {:.3f} +/- {:.3f}   ({})'.format(
        float(np.mean(_ps)), float(np.std(_ps, ddof=1)) if len(_ps) > 1 else 0.0,
        ', '.join(f'{p:.3f}' for p in _ps)))
    # the best seed's checkpoint is what the holdout evaluation below uses
    _best_seed = max(seed_rows.values(), key=lambda r: float(r['psnr']))
    CKPT_DDPM = os.path.join(CKPT_DIR, f"ddpm_seed{_best_seed['seed']}.pth.tar")
    print('  best seed {} -> {}'.format(_best_seed['seed'], os.path.basename(CKPT_DDPM)))

    # Sections 10 and 11 evaluate and visualise ONE model. Bind it here, loaded from the
    # best seed's checkpoint, so those cells work whether this session trained the seeds
    # or merely restored them.
    diffusion = DenoisingDiffusion(config=make_cfg(**BEST_OBJ), device=str(device),
                                   lr=LR_DDPM, checkpoint_path=CKPT_DDPM,
                                   data_parallel=False)
    diffusion.load_checkpoint(CKPT_DDPM)
    print('  loaded for evaluation: epoch {} | best_val {:.4f}'.format(
        diffusion.start_epoch, float(diffusion.best_val_loss)))
else:
    diffusion = None
    print('no seeds trained yet -- sections 10 and 11 will be skipped')


## 9. Loss curves — every seed

In [ ]:
# Histories come from the per-seed checkpoints -- the single-run training cell this used
# to depend on was replaced by the per-seed loop in section 8.
fig, ax = plt.subplots(figsize=(9, 5))
_plotted = 0
for sd in SEEDS_DDPM:
    ck_path = os.path.join(CKPT_DIR, f'ddpm_seed{sd}.pth.tar')
    if not os.path.exists(ck_path):
        continue
    _ck = torch.load(ck_path, map_location='cpu', weights_only=False)
    tr, va = _ck.get('train_losses', []), _ck.get('val_losses', [])
    if not tr:
        continue
    line, = ax.plot(range(1, len(tr) + 1), tr, lw=1.4, label=f'seed {sd} train')
    if va:
        ax.plot(range(1, len(va) + 1), va, lw=1.4, ls='--', color=line.get_color(),
                label=f'seed {sd} val')
        be = int(np.argmin(va)) + 1
        ax.scatter([be], [min(va)], color=line.get_color(), zorder=5, s=28)
    _plotted += 1
    del _ck

if _plotted:
    ax.set_xlabel('epoch'); ax.set_ylabel('DDPM loss')
    ax.set_title(f'Conditional DDPM ({TARGET_SIZE}px) — {BEST_OBJ["prediction_type"]}-prediction, '
                 f'{BEST_OBJ["beta_schedule"]} schedule')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'ddpm_line_emission_loss.png'), dpi=140)
    plt.show()
    print('NOTE: loss is NOT comparable across objectives -- eps and v regress different')
    print('targets and Min-SNR rescales the loss. Compare objectives by PSNR (section 7).')
else:
    plt.close(fig)
    print('no per-seed checkpoints yet -- run section 8 first')


## 10. Validation metrics — PSNR / SSIM / MSE (DDIM-sampled)

`evaluate` DDIM-samples each validation channel from its dirty condition and scores against
clean. Uses EMA weights. Slower than a U-Net forward pass — this is the sampler running per
channel.

In [ ]:
if diffusion is None:
    print('no trained model in memory -- run section 8 first')
    val_metrics_ddpm = None
else:
    val_metrics_ddpm = diffusion.evaluate(ddpm_val_loader, sampling_timesteps=SAMPLING_STEPS, use_ema=True, n_avg=K_AVG)
    print('Validation ({} channels):  PSNR {:.4f} dB | SSIM {:.4f} | MSE {:.6f}'.format(
        val_metrics_ddpm['n'], val_metrics_ddpm['psnr'], val_metrics_ddpm['ssim'], val_metrics_ddpm['mse']))


## 11. Visualize 5 random validation channels — dirty | DDPM denoised | clean

In [ ]:
if diffusion is None:
    print('no trained model in memory -- run section 8 first')
else:
    import random
    from src.evaluation.moment_maps import plot_channel_triptych

    # plot_channel_triptych (src/evaluation/moment_maps.py) replaces plain imshow with no
    # vmin/vmax at all. On a near-empty validation channel matplotlib's autoscale collapses
    # to a near-constant array, which normalises to ~0.5 -- inferno(0.5) is a solid
    # magenta-pink panel, not black. It hit exactly this figure in production (row 3). The
    # shared function asinh-stretches from the CLEAN channel's own range, with a guard for
    # the degenerate case, and is used identically by notebook 05, so the two figures are
    # visually comparable.
    idxs = random.Random(SEED).sample(range(len(val_ds)), 5)
    _rows = []
    for ix in idxs:
        d, c = val_ds[ix]                                     # each (1,H,W), [0,1] shared scale
        pred = diffusion.sample(d[None], sampling_timesteps=SAMPLING_STEPS, use_ema=True, n_avg=K_AVG)[0,0].cpu().numpy()
        ci, ch = val_ds.index[ix]
        _rows.append((d[0].numpy(), pred, c[0].numpy(), f'{val_ds.cube_paths[ci][2]} ch {ch}'))

    fig = plot_channel_triptych(_rows, model_label='DDPM denoised',
                                title='Line-emission DDPM -- validation channels',
                                save_path='../experiments/line_emission_ddpm_comparison.png')
    plt.show()
    print('saved -> experiments/line_emission_ddpm_comparison.png')

## 12. Smoke test — measure DDIM cost before committing to the full run

Section 11 prints nothing until the first batch of `DENOISE_BS_DDPM` channels has run
`SAMPLING_STEPS x K_AVG` = 100 model calls, so **a session that is merely slow is
indistinguishable from one that has hung** — which is how the last two attempts ended with
`denoising run_0002_00560_rt_00 ...` and no further output.

This cell samples one batch, times it, and extrapolates to all 5 held-out cubes. Run it
first; only start section 11 once the projection fits the session budget.


In [ ]:
# Smoke test: is section 11 broken, or just slow? Time one batch and extrapolate.
import time

print('device:', device, '| GPUs:', N_GPU)
if torch.cuda.is_available():
    for i in range(N_GPU):
        _free, _tot = torch.cuda.mem_get_info(i)
        print('  GPU{} {}: {:.1f} / {:.1f} GiB free'.format(
            i, torch.cuda.get_device_name(i), _free / 2**30, _tot / 2**30))
else:
    print('  *** NO GPU ***  DDIM sampling on CPU is ~50x slower -- section 11 cannot finish')
    print('  in a Kaggle session. Set Accelerator to GPU T4 x2 and re-run from section 0.')

_smoke = DenoisingDiffusion(config=cfg, device=str(device), lr=LR_DDPM, checkpoint_path=CKPT_DDPM)
_smoke.load_checkpoint(CKPT_DDPM)

_x = torch.rand(DENOISE_BS_DDPM, 1, TARGET_SIZE, TARGET_SIZE)
_smoke.sample(_x[:1], sampling_timesteps=2, use_ema=True, n_avg=1)   # warm up cuDNN autotune
if torch.cuda.is_available():
    torch.cuda.synchronize()

_t = time.time()
_out = _smoke.sample(_x, sampling_timesteps=SAMPLING_STEPS, use_ema=True, n_avg=K_AVG)
if torch.cuda.is_available():
    torch.cuda.synchronize()
_dt = time.time() - _t

# the sampler is exercised on the real batch shape, so these two catch a broken path
# (wrong shape, NaN from the loaded weights) in seconds instead of hours
assert tuple(_out.shape) == (DENOISE_BS_DDPM, 1, TARGET_SIZE, TARGET_SIZE), _out.shape
assert torch.isfinite(_out).all(), 'sampler produced non-finite output -- checkpoint is bad'
print('\nsampler OK: {} -> {}, range [{:.3f}, {:.3f}]'.format(
    tuple(_x.shape), tuple(_out.shape), float(_out.min()), float(_out.max())))

_C = 201                                   # channels per cube
_batches = -(-_C // DENOISE_BS_DDPM)
_per_cube = _dt * _batches / 60.0
_total_h = _per_cube * len(holdout_cubes) / 60.0
print('one batch of {} channels: {:.1f} s  ({} DDIM steps x K_AVG {} = {} model calls)'.format(
    DENOISE_BS_DDPM, _dt, SAMPLING_STEPS, K_AVG, SAMPLING_STEPS * K_AVG))
print('projected: {:.1f} min/cube  ->  {:.1f} h for {} held-out cubes'.format(
    _per_cube, _total_h, len(holdout_cubes)))

BUDGET_H = 6.0    # headroom inside Kaggle's 9 h session limit
if _total_h > BUDGET_H:
    print('\n*** TOO SLOW: {:.1f} h exceeds the {:.0f} h budget ({:.1f}x over).'.format(
        _total_h, BUDGET_H, _total_h / BUDGET_H))
    print('    Section 11 checkpoints per cube, so it CAN be spread over several sessions')
    print('    as-is. To fit one session instead, cut either knob in section 4 (both linear):')
    print('      K_AVG          {} -> 2   halves cost; posterior mean is noisier'.format(K_AVG))
    print('      SAMPLING_STEPS {} -> 15  cuts ~40%; DDIM holds up well down to ~15'.format(
        SAMPLING_STEPS))
    print('    Then re-run this cell to confirm before starting section 11.')
else:
    print('\nOK: {:.1f} h fits the {:.0f} h budget. Run section 11.'.format(_total_h, BUDGET_H))

del _smoke, _x, _out
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 13. Holdout evaluation of the newly trained model

Same five cubes, same signal-masked metric, same code path as the 6d baseline -- the only
difference is which weights are loaded. If section 8 has not trained anything yet, this
skips and the 6d table stands as the session's result.


In [ ]:
# The improved model: whatever section 8 trained, scored with the identical code path
# the baseline used in 6d, so any difference between the two tables is the model.
IMPROVED_CSV = os.path.join(OUT_DIR, 'moment_map_holdout_summary_ddpm.csv')

if os.path.exists(CKPT_DDPM) and 'seed_rows' in globals() and seed_rows:
    improved_rows, ddpm_maps = run_holdout(CKPT_DDPM, BEST_OBJ, 'improved', IMPROVED_CSV)
else:
    improved_rows = None
    print('section 8 has not trained a seed yet -- nothing new to score.')
    print('The 6d baseline table above is this session\'s result.')

if baseline_rows and improved_rows:
    print('\n' + '=' * 74)
    print('BASELINE vs IMPROVED  (identical cubes, identical metric)')
    print('=' * 74)
    print('{:<12}{:>14}{:>14}{:>14}'.format('', 'M0 (%)', 'M1 (%)', 'M2 (%)'))
    for lbl, rws in (('baseline', baseline_rows), ('improved', improved_rows)):
        vals = []
        for m in ['M0', 'M1', 'M2']:
            v = [r['imp_' + m] for r in rws if r['imp_' + m] == r['imp_' + m]]
            vals.append(float(np.mean(v)) if v else float('nan'))
        print('{:<12}{:>14.1f}{:>14.1f}{:>14.1f}'.format(lbl, *vals))
    print('=' * 74)
    print('Both rows are single models on 5 cubes drawn from only 3 distinct simulations,')
    print('so treat a small gap as unresolved -- see the seed spread in section 8.')


## 13b. Diagnostics — five questions, none of which need retraining

v13 scored PSNR 38.180 with M0 **−56.1% ± 152.2**, the disk clearly recovered in the moment
figure and a constant pedestal over the empty sky. Its own smoke test recorded the sampler
spanning `[0.348, 0.701]` rather than `[0, 1]`.

All five checkpoints persisted, so every question below is inference or post-processing on
weights that already exist:

| arm | question |
|---|---|
| `kavg1` | does averaging 4 reverse draws collapse the output toward the conditional mean? |
| `rescaled` | is only the absolute level wrong, not the structure? |
| `kavg1_rescaled` | both together |
| `patch_model` | the patch arm changed the *training view* — does it fail the same way? |
| `tiled_native` | v13 denoised at 256 and upsampled to 600, capping every number it reports. This runs the model at native 600 with Hann-blended tiles instead. |

Each arm re-scores on the same five cubes and writes its CSV per cube, so a timeout costs
one cube and the next session resumes. Roughly 1 h each, except `tiled_native` at ~2 h — it
samples 9 tiles per channel instead of 1.

The one thing here that **does** need retraining is a second seed, so it is not in this
list. It would measure variance in a result whose mean may yet turn out to be a
de-normalisation bug; that is worth knowing only after these five.

In [ ]:
RUN_DIAGNOSTICS = True

# Ordered by value per GPU-hour. Drop names from this list to shorten a session; the CSVs
# resume per cube, so stopping early costs nothing already computed.
DIAG_ARMS = [
    ('kavg1',          dict(k_avg=1, rescale=False)),
    ('rescaled',       dict(k_avg=4, rescale=True)),
    ('kavg1_rescaled', dict(k_avg=1, rescale=True)),
    ('patch_model',    dict(k_avg=4, rescale=True, ckpt='ddpm_sweep_patch')),
    ('tiled_native',   dict(k_avg=4, rescale=True, tiled=True)),
]


def _find_ckpt(stem):
    """Locate a named checkpoint locally or in any attached input.

    The patch model is `ddpm_sweep_patch`, which section 6b deliberately EXCLUDES from
    restore -- sweep checkpoints must never be picked as the main model, since their loss
    is on a different scale. Scoring one on purpose is a different matter, so it is fetched
    by name here.
    """
    for ext in ('.pth.tar', '.pth', '.ckpt'):
        local = os.path.join(CKPT_DIR, stem + ext)
        if os.path.exists(local):
            return local
    # '.ckpt' as well: a checkpoint that arrives via a DATASET must be renamed, since
    # Kaggle unpacks '.pth' into a directory torch.load rejects (RULES.md #3).
    hits = [h for h in (glob.glob(f'/kaggle/input/**/{stem}.pth*', recursive=True) +
                        glob.glob(f'/kaggle/input/**/{stem}.ckpt', recursive=True))
            if os.path.isfile(h)]
    if hits:
        dst = os.path.join(CKPT_DIR, stem + '.pth.tar')
        shutil.copy2(hits[0], dst)
        print(f'  fetched {stem} from {hits[0]}')
        return dst
    return None


diag_summary = {}
if RUN_DIAGNOSTICS and os.path.exists(CKPT_DDPM):
    _orig = (K_AVG, RESCALE_TO_DIRTY, TILED_NATIVE)
    for _tag, _opt in DIAG_ARMS:
        _ckpt = _find_ckpt(_opt['ckpt']) if _opt.get('ckpt') else CKPT_DDPM
        if _ckpt is None:
            print(f'\n--- {_tag}: SKIPPED, {_opt["ckpt"]} not found ---')
            continue
        _csv = os.path.join(OUT_DIR, f'moment_map_holdout_{_tag}_ddpm.csv')
        print(f'\n{"="*74}\n=== {_tag}: K_AVG={_opt["k_avg"]} rescale={_opt["rescale"]} '
              f'tiled={_opt.get("tiled", False)} ckpt={os.path.basename(_ckpt)}'
              f'\n{"="*74}', flush=True)
        # denoise_cube_ddpm reads all three as globals at call time, so rebinding them here
        # is what selects the arm. Restored in `finally` whatever happens.
        K_AVG = _opt['k_avg']
        RESCALE_TO_DIRTY = _opt['rescale']
        TILED_NATIVE = _opt.get('tiled', False)
        try:
            _rows, _ = run_holdout(_ckpt, BEST_OBJ, _tag, _csv)
        finally:
            K_AVG, RESCALE_TO_DIRTY, TILED_NATIVE = _orig
        diag_summary[_tag] = {m: float(np.mean([r['imp_' + m] for r in _rows]))
                              for m in ('M0', 'M1', 'M2')}
        persist_ckpt(_csv)

    print('\n' + '=' * 74)
    print('{:<20}{:>14}{:>14}{:>14}'.format('arm', 'M0 (%)', 'M1 (%)', 'M2 (%)'))
    print('-' * 74)
    if improved_rows:
        _b = {m: float(np.mean([r['imp_' + m] for r in improved_rows])) for m in ('M0','M1','M2')}
        print('{:<20}{:>14.1f}{:>14.1f}{:>14.1f}'.format('v13 as-run', _b['M0'], _b['M1'], _b['M2']))
    for _tag, _v in diag_summary.items():
        print('{:<20}{:>14.1f}{:>14.1f}{:>14.1f}'.format(_tag, _v['M0'], _v['M1'], _v['M2']))
    print('-' * 74)
    print('{:<20}{:>14.1f}{:>14.1f}{:>14.1f}'.format('U-Net V12 (ref)', 69.8, 17.5, 20.1))
    print('=' * 74)
    print('If `rescaled` recovers M0 and `kavg1` does not, the failure was the output level')
    print('and not the model. If neither does, the model is genuinely wrong about these')
    print('cubes and the pedestal was a symptom. `patch_model` and `tiled_native` say')
    print('whether the training view or the 600->256->600 round trip made it worse.')
else:
    print('diagnostics skipped (RUN_DIAGNOSTICS=False or no checkpoint)')

## 14. Moment maps — clean vs dirty vs DDPM, side by side

The percentages in section 13 compress three 2D scientific products into one number each.
This shows the maps themselves for one held-out cube, on a shared colour scale per moment
set by the **clean** map, so no panel is flattered by autoscaling.

M1 uses a diverging colormap centred on the clean map's median: velocity is signed, and a
sequential map would hide the rotation pattern that makes M1 meaningful.


In [ ]:
MAP_CUBE = holdout_cubes[0]['folder']
_maps_path = os.path.join(OUT_DIR, 'ddpm_moment_maps.npz')

if 'ddpm_maps' in globals() and ddpm_maps is not None:
    _mm = ddpm_maps
elif os.path.exists(_maps_path):
    _z = np.load(_maps_path)
    _mm = {'clean': [_z['c0'], _z['c1'], _z['c2']],
           'dirty': [_z['d0'], _z['d1'], _z['d2']],
           'ddpm':  [_z['n0'], _z['n1'], _z['n2']]}
else:
    _mm = None
    print('no stored moment maps -- run section 13 first (it saves them for this cell)')

if _mm is not None:
    rows_show = [('clean (truth)', _mm['clean']), ('dirty (input)', _mm['dirty']),
                 ('DDPM denoised', _mm['ddpm'])]
    names = ['Moment 0 (intensity)', 'Moment 1 (velocity)', 'Moment 2 (dispersion)']
    cmaps = ['inferno', 'RdBu_r', 'viridis']

    # One colour scale per moment, fixed by the CLEAN map. Autoscaling each panel would
    # make a wrong reconstruction look correct by rescaling its own errors away.
    scales = []
    for j in range(3):
        ref = np.asarray(_mm['clean'][j], float)
        fin = ref[np.isfinite(ref)]
        if j == 1:
            c = float(np.median(fin)); half = float(np.percentile(np.abs(fin - c), 98))
            scales.append((c - half, c + half))
        else:
            scales.append((float(np.percentile(fin, 1)), float(np.percentile(fin, 99))))

    fig, axes = plt.subplots(len(rows_show), 3, figsize=(11, 3.1 * len(rows_show)))
    for r, (lbl, maps) in enumerate(rows_show):
        for j in range(3):
            axm = axes[r, j]
            im = axm.imshow(maps[j], origin='lower', cmap=cmaps[j],
                            vmin=scales[j][0], vmax=scales[j][1])
            axm.set_xticks([]); axm.set_yticks([])
            if r == 0:
                axm.set_title(names[j], fontweight='bold')
            if j == 0:
                axm.set_ylabel(lbl, fontsize=9)
            if r == len(rows_show) - 1:
                fig.colorbar(im, ax=axes[:, j].tolist(), fraction=0.02, pad=0.02)
    fig.suptitle(f'DDPM moment maps — {MAP_CUBE}', fontweight='bold', y=0.995)
    _p = os.path.join(OUT_DIR, 'ddpm_moment_maps.png')
    plt.savefig(_p, dpi=140, bbox_inches='tight'); plt.show()
    print('saved ->', _p)


## 15. Persist DDPM checkpoint to /kaggle/working (survives session end)

Copies the best DDPM checkpoint to the **top level** of `/kaggle/working` (outside the git
clone) so `kaggle kernels output <slug>` retrieves it directly. Save the notebook version
afterward so the `.pth.tar` is a downloadable output artifact.

In [ ]:
import shutil, glob

# v11 trained four sweep arms plus the full model and persisted exactly ONE file, so the
# other five died with the container -- the entire reason this run starts from scratch.
# Everything trained is kept now, and kept as '.pth': cell 6b globs that, and a notebook
# OUTPUT is mounted verbatim, so the extension is free to be the convenient one.
if ON_KAGGLE:
    n_ck = n_res = 0
    for _src in sorted(glob.glob(os.path.join(CKPT_DIR, '*.pth.tar'))):
        # ddpm_seed42.pth.tar -> ddpm_seed42.pth
        _dst = os.path.join('/kaggle/working',
                            os.path.basename(_src)[:-4] if _src.endswith('.pth.tar')
                            else os.path.basename(_src))
        shutil.copy2(_src, _dst); n_ck += 1
        print('  checkpoint -> {} ({:.0f} MB)'.format(os.path.basename(_dst),
                                                      os.path.getsize(_dst) / 1e6))
    for _pat in ('*.csv', '*.png', '*.npz'):
        for _src in sorted(glob.glob(os.path.join(OUT_DIR, _pat))):
            shutil.copy2(_src, os.path.join('/kaggle/working', os.path.basename(_src)))
            n_res += 1
    print('\npersisted {} checkpoint(s) and {} result file(s) to /kaggle/working'.format(
        n_ck, n_res))
    if n_ck == 0:
        print('  WARNING: no checkpoints in ' + CKPT_DIR + ' -- nothing to resume from')
    print('\nTo resume: Add Input -> Notebooks -> THIS notebook. Do NOT re-upload a .pth as')
    print('a Dataset -- a torch checkpoint is internally a zip, Kaggle unpacks it, and it')
    print('arrives as a directory that torch.load rejects with "Is a directory".')
else:
    print('not on Kaggle -- skip persist')

## Collect this notebook's outputs

Every artifact this notebook produced, copied into a **versioned run folder** together with
a manifest recording the commit, the UTC time and a checksum per file:

    <outputs>/06-ddpm-line-emission/<UTC timestamp>_<git sha>/

Nothing overwrites anything, so two runs can be compared directly. Results used to land in
one flat `results/` directory with no record of which notebook or which revision wrote them
-- a moment CSV from before the M2 noise-clip fix was indistinguishable from one after it.

On Kaggle this writes to `/kaggle/working/outputs/`, at the top level, so the bundle is
unambiguously part of the notebook Output rather than buried in the git clone. Download that
folder and commit it under `DENOISING_DIFFUSION/results/`.


In [ ]:
from src.evaluation.collect_outputs import collect_outputs

# Checkpoints are deliberately NOT listed. Section 15 already copies all six to the top
# level of /kaggle/working, and this collector writes INTO /kaggle/working/outputs -- so
# naming them here would put a second ~2 GB copy inside the same notebook Output.
_run_dir = collect_outputs(
    '06-ddpm-line-emission',
    [
        'ddpm_objective_sweep.csv',              # section 7: all four arms
        'ddpm_seed_repeats.csv',                 # section 8
        'moment_map_holdout_baseline_ddpm.csv',  # 6d, if a checkpoint was restored
        'moment_map_holdout_summary_ddpm.csv',   # section 13: the headline table
        'moment_map_holdout_summary_ddpm.png',
        'ddpm_moment_maps.npz',                  # maps for cube 1, so the figure can be redrawn
        'ddpm_moment_maps.png',
        'ddpm_line_emission_loss.png',
        'line_emission_ddpm_comparison.png',
    ],
    extra={'objective': BEST_OBJ, 'epochs': EPOCHS_DDPM, 'seeds': SEEDS_DDPM,
           'sampling_steps': SAMPLING_STEPS, 'k_avg': K_AVG,
           'sweep_epochs': SWEEP_EPOCHS if RUN_SWEEP else None,
           'target_size': TARGET_SIZE, 'tiled_native': TILED_NATIVE},
)
print('\nAnything listed as NOT FOUND above did not get written this run -- check why '
      'before quoting the run as complete.')